# Black-Litterman Crypto — Backtest Walk-Forward

**Parâmetros:** lookback=365 dias | holding=7 dias | universo fixo (11 ativos)

**Estratégias comparadas:**
| # | Estratégia | Descrição |
|---|---|---|
| 1 | Equal-Weight | 1/N — benchmark trivial |
| 2 | Market-Cap | Pesos por market cap |
| 3 | Markowitz Puro | MV clássico com μ histórico |
| 4 | BL Neutro | Black-Litterman sem views |
| 5 | BL + RSI | BL + views absolutas via RSI(14) |
| 6 | BL + Momentum | BL + view relativa top3 vs bottom3 |
| 7 | BL + RSI + Momentum | BL + views combinadas |

> ⏱️ A célula de backtest leva alguns minutos. Execute célula por célula.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.config import (
    ARQUIVO_RETORNOS, ARQUIVO_PRECOS, ARQUIVO_MARKET_CAP, DIR_FIGURAS
)
from src.backtest import WalkForwardBacktest
from src.estrategias import (
    EqualWeight, MarketCapWeight, MarkowitzPuro,
    BlackLittermanNeutro, BlackLittermanRSI,
    BlackLittermanMomentum, BlackLittermanCombinado,
)
from src.visualizacao import (
    plotar_equity_curves, plotar_drawdowns, plotar_rolling_sharpe,
    plotar_heatmap_pesos, plotar_distribuicao_retornos, tabela_metricas,
)
from src.portfolio_utils import DIAS_ANO_CRIPTO

sns.set_theme(style='whitegrid', palette='tab10')
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup OK')

Setup OK


## 1. Setup e carregamento dos dados

In [2]:
retornos = pd.read_parquet(ARQUIVO_RETORNOS)
precos   = pd.read_parquet(ARQUIVO_PRECOS)
mc_df    = pd.read_parquet(ARQUIVO_MARKET_CAP)
mc_df    = mc_df.set_index('ticker') if 'ticker' in mc_df.columns else mc_df
market_caps = mc_df['market_cap_usd']

ativos = retornos.columns.intersection(precos.columns).intersection(market_caps.index)
retornos    = retornos[ativos]
precos      = precos[ativos]
market_caps = market_caps[ativos]

LOOKBACK = 365
HOLDING  = 7
LAM      = 2.5
TAU      = 0.05
P_MAX    = 0.40

print(f'Ativos  : {list(ativos)}')
print(f'Período : {retornos.index.min().date()} → {retornos.index.max().date()}')
print(f'Shape   : {retornos.shape}')

OSError: Repetition level histogram size mismatch

In [ ]:
backtest = WalkForwardBacktest(
    retornos=retornos, precos=precos,
    market_caps_atuais=market_caps,
    lookback_days=LOOKBACK, holding_period_days=HOLDING,
)

kw = dict(market_caps=market_caps, risk_aversion=LAM, tau=TAU, peso_maximo=P_MAX)
estrategias = {
    'Equal-Weight':        EqualWeight(),
    'Market-Cap':          MarketCapWeight(market_caps),
    'Markowitz Puro':      MarkowitzPuro(LAM, P_MAX),
    'BL Neutro':           BlackLittermanNeutro(**kw),
    'BL + RSI':            BlackLittermanRSI(**kw),
    'BL + Momentum':       BlackLittermanMomentum(**kw),
    'BL + RSI + Momentum': BlackLittermanCombinado(**kw),
}

datas = backtest.gerar_datas_rebalanceamento()
print(f'Rebalanceamentos: {len(datas)} | {datas[0].date()} → {datas[-1].date()}')

In [ ]:
# ⏱️ Esta célula leva alguns minutos
print('Executando backtest...')
resultados = backtest.comparar_estrategias(estrategias)
print('Concluído!')

## 2. Tabela de métricas comparativas

In [ ]:
df_met = tabela_metricas(resultados)
try:
    from IPython.display import Markdown, display
    display(Markdown(df_met.to_markdown()))
except Exception:
    display(df_met)

## 3. Equity Curves (linear e log)

In [ ]:
_ = plotar_equity_curves(resultados)
plt.show()
_ = plotar_equity_curves(resultados, titulo='Equity Curves (escala log)', log_scale=True)
plt.show()

## 4. Drawdowns

In [ ]:
_ = plotar_drawdowns(resultados)
plt.show()

## 5. Rolling Sharpe (janela 90 dias)

In [ ]:
_ = plotar_rolling_sharpe(resultados, janela_dias=90)
plt.show()

## 6. Heatmap de pesos — estratégias Black-Litterman

In [ ]:
for nome in ['BL Neutro', 'BL + RSI', 'BL + Momentum', 'BL + RSI + Momentum']:
    if nome in resultados:
        _ = plotar_heatmap_pesos(resultados[nome])
        plt.show()

## 7. Comparação de turnover

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
turnover_medio = {nome: res.turnover().mean() for nome, res in resultados.items()}
nomes = list(turnover_medio.keys())
vals  = list(turnover_medio.values())
bars = ax.bar(nomes, vals, color=sns.color_palette('tab10', len(nomes)), alpha=0.85)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_title('Turnover Médio por Estratégia', fontsize=13)
ax.set_ylabel('Turnover (0=sem troca, 1=renovação total)')
ax.set_xticklabels(nomes, rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. Distribuição dos retornos diários

In [ ]:
_ = plotar_distribuicao_retornos(resultados)
plt.show()

## 9. Sensibilidade: variar tau

In [ ]:
taus = [0.025, 0.05, 0.10, 0.20]
res_tau = {}
for t in taus:
    est = BlackLittermanRSI(market_caps=market_caps, risk_aversion=LAM, tau=t, peso_maximo=P_MAX)
    res_tau[f'BL+RSI tau={t}'] = backtest.executar_estrategia(est)

df_tau = tabela_metricas(res_tau)
print('Sensibilidade tau:')
try:
    from IPython.display import Markdown, display
    display(Markdown(df_tau.to_markdown()))
except Exception:
    display(df_tau)

In [ ]:
_ = plotar_equity_curves(res_tau, titulo='Sensibilidade: tau')
plt.show()

## 10. Sensibilidade: variar risk_aversion (λ)

In [ ]:
lambdas = [1.5, 2.5, 3.5, 5.0]
res_lam = {}
for lam in lambdas:
    est = BlackLittermanRSI(market_caps=market_caps, risk_aversion=lam, tau=TAU, peso_maximo=P_MAX)
    res_lam[f'BL+RSI λ={lam}'] = backtest.executar_estrategia(est)

df_lam = tabela_metricas(res_lam)
print('Sensibilidade λ (risk_aversion):')
try:
    from IPython.display import Markdown, display
    display(Markdown(df_lam.to_markdown()))
except Exception:
    display(df_lam)

_ = plotar_equity_curves(res_lam, titulo='Sensibilidade: risk_aversion (λ)')
plt.show()

## 11. Discussão final

*(Preencha após visualizar os resultados com seus dados reais)*

### Qual estratégia teve melhor Sharpe?
...

### Black-Litterman com views supera o neutro?
...

### Markowitz puro é instável conforme esperado?
...

### Limitações observadas
1. **Market caps estáticos**: os pesos de equilíbrio são calculados com o market cap atual, não o histórico de cada data de rebalanceamento. Isso introduz um viés de lookahead leve.
2. **Sem custos de transação**: o turnover observado nas estratégias BL é real e geraria custos não modelados.
3. **RSI threshold fixo**: os thresholds 30/70 são arbitrários; uma calibração in-sample poderia melhorar resultados, mas introduziria overfitting.
4. **Período curto**: 2 anos de dados incluem apenas um ciclo de mercado. Conclusões sobre superioridade de estratégia devem ser cautelosas.